# Create Survival Prediction Model using PyTorch

One strategy would be to replicate the [DeepSurv model](https://arxiv.org/abs/1606.00931), which is a deep neural network for survival analysis, essentially a nonlinear version of the Cox proportional hazard model. 

DeepSurv takes a list of features, and learns a risk function from input features. 
No restrictions, all types of data: clinical, gene expression, genetic mutations, are all treated the same. 

## Multi-modal Learning Model

Our data includes 3 unique modalities: 
- Clinical metadata
- Z-scored Gene Expression 
- Genetic Mutation data

So we will create separate encoders for each modality

Clinical features ──► Clinical MLP    ┐

Expression matrix ──► Expr Encoder ├─► Fusion ─► Risk score ─► Cox loss

Mutation matrix ────► Mut Encoder ─┘

## Architecture
1. Clinical Encoder (MLP): Low depth, minimal regularization
2. Gene Expression Encoder (Autoencoder or Bottleneck MLP)
Options: 
- Variance Filtered Genes
- Pathway Scores
- Autoencoder Latent Space
3. Mutational Autoencoder
- Binary gene-level mutation matrix
- Tumor mutational burden

## Training strategy 

1. Train clinical-only DeepSurv model
2. Add expression autoencoder (similar to Cox model) 
3. Add mutation autoencoder 
4. Fine-tune all layers

## Evaluation Strategies
Stratify patients by predicted risk tertiles

Plot Kaplan Meier curves

Compare against one another:

CoxPH (clinical)

CoxPH (clinical + expression)

DeepSurv multimodal

In [9]:
# load libraries
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pyhere as here


### Load Data

In [12]:
### Load data 

# load clinical metadata
clinical = pd.read_csv(here.here("data", "processed","clinical_data.csv"))
# load expression data
expression_data = pd.read_csv(here.here("data", "processed","expression_data.csv"))
# mutation binary data
mutation_binary = pd.read_csv(here.here("data", "processed","mutation_binary_data.csv"))
# mutation classified data
mutation_classified = pd.read_csv(here.here("data", "processed","mutation_classified_data.csv"))

# Example of Multi-modal Learning Model

Mockup of what the final model will look like, with toy data

### Loss Function

In [14]:
# partial likelihood loss function for Cox Proportional Hazards model
def cox_ph_loss(risk_scores, times, events):
    """
    Negative partial log-likelihood for Cox PH
    """
    order = torch.argsort(times, descending=True)
    risk_scores = risk_scores[order]
    events = events[order]

    log_cumsum = torch.logcumsumexp(risk_scores, dim=0)
    loss = -torch.sum((risk_scores - log_cumsum) * events)
    return loss / events.sum()

### Modality-specific Encoders

In [16]:
import torch.nn as nn

# clinical data encoder
class ClinicalEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

In [17]:
# Expression data encoder
class ExpressionEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

In [19]:
# Mutation data encoder
class MutationEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

### Multimodal DeepServ Model

In [ ]:
# Multi-modal DeepSurv model, combining clinical, expression, and mutation data
class MultiModalDeepSurv(nn.Module):
    def __init__(self, clin_dim, expr_dim, mut_dim):
        super().__init__()

        self.clin_enc = ClinicalEncoder(clin_dim)
        self.expr_enc = ExpressionEncoder(expr_dim)
        self.mut_enc  = MutationEncoder(mut_dim)

        self.head = nn.Sequential(
            nn.Linear(32 + 128 + 64, 64),
            nn.ReLU(),
            nn.Dropout(0.3), # regularization, prevents overfitting by setting 30% of neurons to zero during training
            nn.Linear(64, 1)
        )

    def forward(self, x_clin, x_expr, x_mut):
        z_clin = self.clin_enc(x_clin)
        z_expr = self.expr_enc(x_expr)
        z_mut  = self.mut_enc(x_mut)

        z = torch.cat([z_clin, z_expr, z_mut], dim=1)
        return self.head(z).squeeze(-1)


# Training function for one epoch
def train_epoch(model, optimizer, x_clin, x_expr, x_mut, time, event):
    model.train()
    optimizer.zero_grad()

    risk = model(x_clin, x_expr, x_mut)
    loss = cox_ph_loss(risk, time, event)

    loss.backward()
    optimizer.step()
    return loss.item()


### Example run with randomized data

In [23]:
# example run with randomized data
N = 1500  # number of samples
C = 30      # clinical
G = 1000   # expression
M = 200    # mutation

x_clin = torch.randn(N, C)
x_expr = torch.randn(N, G)
x_mut  = torch.randint(0, 2, (N, M)).float()

time  = torch.rand(N) * 100
event = torch.randint(0, 2, (N,)).float()

model = MultiModalDeepSurv(C, G, M)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(50):
    loss = train_epoch(
        model,
        optimizer,
        x_clin,
        x_expr,
        x_mut,
        time,
        event
    )
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss:.4f}")


Epoch 0 | Loss: 6.3051
Epoch 10 | Loss: 5.7033
Epoch 20 | Loss: 4.6340
Epoch 30 | Loss: 4.5254
Epoch 40 | Loss: 4.3099


### Evaluation (C-index)

In [24]:
from lifelines.utils import concordance_index

model.eval()
with torch.no_grad():
    risk = model(x_clin, x_expr, x_mut).numpy()

c_index = concordance_index(
    time.numpy(),
    -risk,
    event.numpy()
)

print("C-index:", c_index)


C-index: 0.9827553716259907


Remarkably good C-index! (Almost like the data isn't real! )

Next, we will train the model in increments, beginning with the clinical encoder only.

# Clinical-only Model

To begin, we will train a neural Cox model using only clinical variables, similar to the original Cox Hazard model. 

Once we have an effective model using only clinical features, we can expand it to clinical + gene expression, and finally, clinical, gene expression, and mutation data. 

## Clinical DeepSurv model class
This is a single neural network with a sequence of:
- Linear layer (input -> 32 dim)
- ReLu
- 20% dropout normalization
- Final linear layer (32 dim -> risk score)

### First: Encode clinical data for use with PyTorch

Clinical data includes categorical/ string based data, while PyTorch expects data as tensors. 

We will one-hot encode categorical data. 

In [34]:
import pandas as pd
pd.set_option('display.max_columns', None)

# overview of our clinical data
clinical.head()

,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_+_claudin-low_subtype,cohort,er_status_measured_by_ihc,er_status,neoplasm_histologic_grade,her2_status_measured_by_snp6,her2_status,tumor_other_histologic_subtype,hormone_therapy,inferred_menopausal_state,integrative_cluster,primary_tumor_laterality,lymph_nodes_examined_positive,mutation_count,nottingham_prognostic_index,oncotree_code,overall_survival_months,overall_survival,pr_status,radio_therapy,3-gene_classifier_subtype,tumor_size,tumor_stage,death_from_cancer
0,0,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,0,claudin-low,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,1,Post,4ER+,Right,10.0,NaN,6.044,IDC,140.500000,1,Negative,1,ER-/HER2-,22.0,2.0,Living
1,2,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumA,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,1,Pre,4ER+,Right,0.0,2.0,4.020,IDC,84.633333,1,Positive,1,ER+/HER2- High Prolif,10.0,1.0,Living
2,5,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,1,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Ductal/NST,1,Pre,3,Right,1.0,2.0,4.030,IDC,163.700000,0,Positive,0,unknown,15.0,2.0,Died of Disease
3,6,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,1,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Mixed,1,Pre,9,Right,3.0,1.0,4.050,MDLC,164.933333,1,Positive,1,unknown,25.0,2.0,Living
4,8,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,1,LumB,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Mixed,1,Post,9,Right,8.0,2.0,6.080,MDLC,41.366667,0,Positive,1,ER+/HER2- High Prolif,40.0,2.0,Died of Disease


In [35]:
# define outcome variables
df = clinical.copy()
time_col = "overall_survival_months"
event_col = "death_from_cancer"

# 'event' will be 1 if died of cancer of disease, 0 otherwise (including died of other causes or alive)
df["event"] = (df[event_col] == "Died of Disease").astype(int)
df["time"] = df[time_col]


In [69]:
df.head()

,patient_id,age_at_diagnosis,type_of_breast_surgery,cancer_type,cancer_type_detailed,cellularity,chemotherapy,pam50_+_claudin-low_subtype,cohort,er_status_measured_by_ihc,er_status,neoplasm_histologic_grade,her2_status_measured_by_snp6,her2_status,tumor_other_histologic_subtype,hormone_therapy,inferred_menopausal_state,integrative_cluster,primary_tumor_laterality,lymph_nodes_examined_positive,mutation_count,nottingham_prognostic_index,oncotree_code,overall_survival_months,overall_survival,pr_status,radio_therapy,3-gene_classifier_subtype,tumor_size,tumor_stage,death_from_cancer,event,time,cancer_type_detailed_clean,tumor_other_histologic_subtype_clean
0,0,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,0,claudin-low,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,1,Post,4ER+,Right,10.0,NaN,6.044,IDC,140.500000,1,Negative,1,ER-/HER2-,22.0,2.0,Living,0,140.500000,Breast Invasive Ductal Carcinoma,Ductal/NST
1,2,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,0,LumA,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,1,Pre,4ER+,Right,0.0,2.0,4.020,IDC,84.633333,1,Positive,1,ER+/HER2- High Prolif,10.0,1.0,Living,0,84.633333,Breast Invasive Ductal Carcinoma,Ductal/NST
2,5,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,1,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Ductal/NST,1,Pre,3,Right,1.0,2.0,4.030,IDC,163.700000,0,Positive,0,unknown,15.0,2.0,Died of Disease,1,163.700000,Breast Invasive Ductal Carcinoma,Ductal/NST
3,6,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,1,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Mixed,1,Pre,9,Right,3.0,1.0,4.050,MDLC,164.933333,1,Positive,1,unknown,25.0,2.0,Living,0,164.933333,Breast Mixed Ductal and Lobular Carcinoma,Mixed
4,8,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,1,LumB,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Mixed,1,Post,9,Right,8.0,2.0,6.080,MDLC,41.366667,0,Positive,1,ER+/HER2- High Prolif,40.0,2.0,Died of Disease,1,41.366667,Breast Mixed Ductal and Lobular Carcinoma,Mixed


In [70]:
df[["3-gene_classifier_subtype"]].value_counts()

3-gene_classifier_subtype
ER+/HER2- Low Prolif         619
ER+/HER2- High Prolif        603
ER-/HER2-                    290
unknown                      204
HER2+                        188
Name: count, dtype: int64

In [71]:
# group rare cancer types into "Other" category
# This will make one hot encoding and modeling more robust by reducing the number of categories with very few samples
rare_threshold = 50

value_counts = df["cancer_type_detailed"].value_counts()

rare_classes = value_counts[value_counts < rare_threshold].index

df["cancer_type_detailed_clean"] = df["cancer_type_detailed"].replace(
    rare_classes, "Other"
)

# do same for histologic subtype
value_counts = df["tumor_other_histologic_subtype"].value_counts()

rare_classes = value_counts[value_counts < rare_threshold].index

df["tumor_other_histologic_subtype_clean"] = df["tumor_other_histologic_subtype"].replace(
    rare_classes, "Other"
)

In [72]:
df[["cancer_type_detailed","cancer_type_detailed_clean"]].value_counts()

cancer_type_detailed                       cancer_type_detailed_clean               
Breast Invasive Ductal Carcinoma           Breast Invasive Ductal Carcinoma             1500
Breast Mixed Ductal and Lobular Carcinoma  Breast Mixed Ductal and Lobular Carcinoma     207
Breast Invasive Lobular Carcinoma          Breast Invasive Lobular Carcinoma             142
Breast Invasive Mixed Mucinous Carcinoma   Other                                          22
Breast                                     Other                                          17
Metaplastic Breast Cancer                  Other                                           1
Name: count, dtype: int64

In [73]:
df[["tumor_other_histologic_subtype","tumor_other_histologic_subtype_clean"]].value_counts()

tumor_other_histologic_subtype  tumor_other_histologic_subtype_clean
Ductal/NST                      Ductal/NST                              1454
Mixed                           Mixed                                    207
Lobular                         Lobular                                  142
Medullary                       Other                                     25
Mucinous                        Other                                     22
Tubular/ cribriform             Other                                     21
Other                           Other                                     17
Metaplastic                     Other                                      1
Name: count, dtype: int64

In [ ]:
# Drop identifiers, outcome columns, and pre-modified columns from clinical features
drop_cols = [
    "patient_id",
    "cancer_type",
    "cohort",
    "cancer_type_detailed",
    "er_status_measured_by_ihc",
    "her2_status_measured_by_snp6",
    "tumor_other_histologic_subtype",
    "integrative_cluster",
    "oncotree_code",
    time_col,
    event_col
]

# X will now contain only clinical features that will go into model training
X = df.drop(columns=drop_cols + ["event", "time"])
y_time = df["time"].values
y_event = df["event"].values

#### Identify feature types

In [ ]:
# different data types in clinical data
X.dtypes.value_counts()

object     12
float64     6
int64       4
Name: count, dtype: int64

In [84]:
# identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

# fill missing categorical data with "Unknown"
X[categorical_cols] = X[categorical_cols].fillna("Unknown")

In [85]:
numeric_cols

['age_at_diagnosis',
 'chemotherapy',
 'neoplasm_histologic_grade',
 'hormone_therapy',
 'lymph_nodes_examined_positive',
 'mutation_count',
 'nottingham_prognostic_index',
 'overall_survival',
 'radio_therapy',
 'tumor_size']

In [86]:
categorical_cols

['type_of_breast_surgery',
 'cellularity',
 'pam50_+_claudin-low_subtype',
 'er_status',
 'her2_status',
 'inferred_menopausal_state',
 'primary_tumor_laterality',
 'pr_status',
 '3-gene_classifier_subtype',
 'tumor_stage',
 'cancer_type_detailed_clean',
 'tumor_other_histologic_subtype_clean']

#### Train/Validation Split 

In [103]:
from sklearn.model_selection import train_test_split

X_train, X_val, time_train, time_val, event_train, event_val = train_test_split(
    X, y_time, y_event,
    test_size=0.2,
    stratify=y_event,
    random_state=42
)


#### Process numeric features for training and validation set separately
Scaling/imputation depends on the data that we see, so we must normalize the data separately for training and validation to prevent data leakage

In [104]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_num = num_imputer.fit_transform(X_train[numeric_cols])
X_train_num = scaler.fit_transform(X_train_num)

X_val_num = num_imputer.transform(X_val[numeric_cols])
X_val_num = scaler.transform(X_val_num)

#### Process categorical features separately 
One-hot encode separately

In [120]:
X_train_cat = pd.get_dummies(
    X_train[categorical_cols],
    drop_first=True
).astype(float)

X_val_cat = pd.get_dummies(
    X_val[categorical_cols],
    drop_first=True
).astype(float)

X_val_cat = X_val_cat.reindex(
    columns=X_train_cat.columns,
    fill_value=0
)


#### Combine features

In [122]:
import numpy as np

X_train_final = np.hstack([
    X_train_num.astype(np.float32),
    X_train_cat.values.astype(np.float32)
])

X_val_final = np.hstack([
    X_val_num.astype(np.float32),
    X_val_cat.values.astype(np.float32)
])


In [128]:
X_val_final.dtype

dtype('float32')

In [129]:
X_train_final.shape

(1523, 44)

#### Convert to PyTorch Tensors

In [130]:
import torch

X_train_tensor = torch.tensor(X_train_final, dtype=torch.float32)
time_train_tensor = torch.tensor(time_train, dtype=torch.float32)
event_train_tensor = torch.tensor(event_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val_final, dtype=torch.float32)
time_val_tensor = torch.tensor(time_val, dtype=torch.float32)
event_val_tensor = torch.tensor(event_val, dtype=torch.float32)


In [131]:
print(X_val_num.dtype)
print(X_val_cat.dtypes.value_counts())


float64
float64    33
int64       1
Name: count, dtype: int64


#### Clinical Only DeepSurv model class definition

In [132]:
import torch.nn as nn

class ClinicalSurvNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 1)  # risk score
        )

    def forward(self, x):
        return self.net(x).squeeze()


#### Cox partial likelihood loss

In [133]:
def cox_loss(risk, time, event):
    order = torch.argsort(time, descending=True)
    risk = risk[order]
    event = event[order]

    log_cumsum = torch.logcumsumexp(risk, dim=0)
    loss = -torch.sum((risk - log_cumsum) * event) / event.sum()
    return loss


#### Model training

In [134]:
model = ClinicalSurvNet(X_train_final.shape[1])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()

    risk = model(X_train_tensor)
    loss = cox_loss(risk, time_train_tensor, event_train_tensor)

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")


Epoch 0, Loss: 7.2096
Epoch 20, Loss: 5.9174
Epoch 40, Loss: 5.7811
Epoch 60, Loss: 5.6715
Epoch 80, Loss: 5.5670
Epoch 100, Loss: 5.5182
Epoch 120, Loss: 5.4456
Epoch 140, Loss: 5.3914
Epoch 160, Loss: 5.2819
Epoch 180, Loss: 5.2416


#### Evaluate on validation set

In [135]:
from lifelines.utils import concordance_index

model.eval()
with torch.no_grad():
    val_risk = model(X_val_tensor).numpy()

c_index = concordance_index(
    time_val,
    -val_risk,
    event_val
)

print("Validation C-index:", c_index)


Validation C-index: 0.8390565170776535
